In [1]:
!pip install tensorflow_hub resampy soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.3 MB/s eta 0:00:00


In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import csv
import resampy
import soundfile as sf
from IPython.display import Audio, display

# YAMNet 모델 로드 (1~2분 걸림)
print("모델 로드 중...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# 클래스 이름 로드
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
with tf.io.gfile.GFile(class_map_path) as f:
    class_names = [row['display_name'] for row in csv.DictReader(f)]

print(f"✅ 모델 로드 완료! 총 {len(class_names)}개 클래스 인식 가능")

모델 로드 중...
✅ 모델 로드 완료! 총 521개 클래스 인식 가능


In [3]:
# 각 조원의 담당 소리 키워드 설정
SOUND_CATEGORIES = {
    '화재벨':     ['fire', 'alarm', 'siren', 'smoke'],
    '아기울음':   ['baby', 'cry', 'infant'],
    '개소리':     ['dog', 'bark', 'howl', 'growl'],
    '초인종':     ['doorbell', 'ding-dong', 'bell'],
    '현관문':     ['door', 'knock', 'slam'],
    '물소리':     ['water', 'faucet', 'tap', 'drip', 'pour'],
}

def load_and_classify(file_path):
    # 오디오 로드
    wav_data, sample_rate = sf.read(file_path, dtype=np.int16)
    if len(wav_data.shape) > 1:
        wav_data = wav_data.mean(axis=1)
    waveform = wav_data.astype(np.float32) / 32768.0
    if sample_rate != 16000:
        waveform = resampy.resample(waveform, sample_rate, 16000)

    # YAMNet 추론
    scores, embeddings, spectrogram = yamnet_model(waveform)
    mean_scores = scores.numpy().mean(axis=0)
    top_indices = np.argsort(mean_scores)[::-1][:10]

    # 오디오 재생
    print(f"\n🎵 파일: {file_path} ({len(waveform)/16000:.1f}초)")
    display(Audio(waveform, rate=16000))

    # Top-10 결과 출력
    print("\n📊 Top-10 분류 결과:")
    for rank, i in enumerate(top_indices, 1):
        name = class_names[i]
        score = mean_scores[i]
        # 어떤 카테고리에 해당하는지 체크
        matched = ""
        for category, keywords in SOUND_CATEGORIES.items():
            if any(kw in name.lower() for kw in keywords):
                matched = f" ← {category}"
                break
        print(f"  {rank:2d}. {name:<35s} {score:.4f}{matched}")

    # 카테고리별 점수 요약
    print("\n📋 카테고리별 점수:")
    for category, keywords in SOUND_CATEGORIES.items():
        cat_score = sum(
            mean_scores[i] for i in range(len(class_names))
            if any(kw in class_names[i].lower() for kw in keywords)
        )
        bar = "█" * int(cat_score * 50)
        print(f"  {category:<8s} {cat_score:.4f} {bar}")

print("✅ 분류 함수 준비 완료! (전체 카테고리 지원)")

✅ 분류 함수 준비 완료!


In [6]:
from google.colab import files

print("=" * 50)
print("🎧 소리 파일을 업로드하세요!")
print("   wav, mp3, flac 다 가능해요")
print("   여러 파일 한 번에 선택 가능!")
print("=" * 50)

uploaded = files.upload()

for filename in uploaded:
    load_and_classify(filename)

print("\n" + "=" * 50)
print(f"📋 총 {len(uploaded)}개 파일 테스트 완료!")
print("=" * 50)

🎧 소리 파일을 업로드하세요!
   wav, mp3, flac 다 가능해요
   여러 파일 한 번에 선택 가능!


Saving 사이렌1.mp3 to 사이렌1.mp3
Saving 사이렌2.mp3 to 사이렌2.mp3
Saving 사이렌3.mp3 to 사이렌3.mp3
Saving 사이렌4.mp3 to 사이렌4.mp3
Saving 사이렌5.mp3 to 사이렌5.mp3
Saving 사이렌6.mp3 to 사이렌6.mp3
Saving 사이렌7.mp3 to 사이렌7.mp3

🎵 파일: 사이렌1.mp3 (4.3초)



📊 Top-10 분류 결과:
      1. Buzzer                              0.3775
  🔥  2. Alarm                               0.3206
  🔥  3. Fire alarm                          0.1908
  🔥  4. Car alarm                           0.1200
  🔥  5. Smoke detector, smoke alarm         0.1190
      6. Beep, bleep                         0.0730
      7. Squawk                              0.0536
      8. Music                               0.0323
      9. Inside, small room                  0.0321
     10. Animal                              0.0262

🚨 화재/알람 관련 총점: 0.7504
  ✅ 화재벨로 인식 성공!

🎵 파일: 사이렌2.mp3 (9.1초)



📊 Top-10 분류 결과:
  🔥  1. Alarm clock                         0.2548
  🔥  2. Alarm                               0.2192
      3. Telephone bell ringing              0.2051
      4. Telephone                           0.2030
      5. Ringtone                            0.1090
  🔥  6. Fire alarm                          0.0969
      7. Buzzer                              0.0722
      8. Tools                               0.0701
      9. Bicycle bell                        0.0624
     10. Bell                                0.0475

🚨 화재/알람 관련 총점: 0.5710
  ✅ 화재벨로 인식 성공!

🎵 파일: 사이렌3.mp3 (3.5초)



📊 Top-10 분류 결과:
      1. Music                               0.3314
      2. Bell                                0.0951
      3. Musical instrument                  0.0622
      4. Tools                               0.0465
      5. Cymbal                              0.0408
      6. Chime                               0.0356
      7. Cacophony                           0.0297
      8. Power tool                          0.0295
  🔥  9. Alarm                               0.0294
     10. Whistle                             0.0256

🚨 화재/알람 관련 총점: 0.0294
  ❌ 화재벨로 인식 실패 - fine-tuning 필요한 샘플

🎵 파일: 사이렌4.mp3 (3.6초)



📊 Top-10 분류 결과:
      1. Music                               0.3097
  🔥  2. Alarm clock                         0.1306
      3. Ding                                0.1023
  🔥  4. Alarm                               0.0982
      5. Whistle                             0.0917
      6. Buzzer                              0.0907
      7. Glass                               0.0861
  🔥  8. Fire alarm                          0.0474
      9. Chink, clink                        0.0325
     10. Steam whistle                       0.0311

🚨 화재/알람 관련 총점: 0.2761
  ✅ 화재벨로 인식 성공!

🎵 파일: 사이렌5.mp3 (8.7초)



📊 Top-10 분류 결과:
  🔥  1. Alarm clock                         0.3264
  🔥  2. Alarm                               0.2831
      3. Telephone                           0.2542
      4. Telephone bell ringing              0.2365
      5. Ringtone                            0.1481
  🔥  6. Fire alarm                          0.0905
      7. Buzzer                              0.0724
      8. Tools                               0.0696
      9. Bell                                0.0464
     10. Clock                               0.0456

🚨 화재/알람 관련 총점: 0.7000
  ✅ 화재벨로 인식 성공!

🎵 파일: 사이렌6.mp3 (4.5초)



📊 Top-10 분류 결과:
      1. Crying, sobbing                     0.2155
      2. Baby cry, infant cry                0.1903
  🔥  3. Alarm                               0.1196
      4. Whistling                           0.1132
      5. Bird                                0.1057
      6. Laughter                            0.1018
      7. Wild animals                        0.0881
      8. Theremin                            0.0702
      9. Animal                              0.0645
     10. Baby laughter                       0.0629

🚨 화재/알람 관련 총점: 0.1196
  ✅ 화재벨로 인식 성공!

🎵 파일: 사이렌7.mp3 (3.7초)



📊 Top-10 분류 결과:
      1. Crying, sobbing                     0.4224
      2. Baby cry, infant cry                0.3540
      3. Whistle                             0.1341
  🔥  4. Alarm                               0.1283
      5. Sine wave                           0.0989
      6. Theremin                            0.0970
      7. Whimper                             0.0662
      8. Inside, small room                  0.0534
      9. Whistling                           0.0476
     10. Babbling                            0.0435

🚨 화재/알람 관련 총점: 0.1283
  ✅ 화재벨로 인식 성공!

📋 총 7개 파일 테스트 완료!
